*0.3 Classical NLP*

# NER

**The situation.** Tickets must be logged into the CRM with the customer's company, the amount and the date: "Acme Corp was charged $1,250 on 3 March by our London office." Agents copy these by hand, and 5% are wrong. A regex for money works; one for company names does not exist.

**Named-entity recognition.** A model that labels spans of text as PERSON, ORG, MONEY, DATE, GPE (places) and more. spaCy's English pipeline does it in a few milliseconds per ticket, on CPU.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")
ticket = (
    "Acme Corp was charged $1,250 on 3 March by our London office. Priya Shah asked for a refund "
    "within 14 days."
)
document = nlp(ticket)
print(f"{'text':<14}{'label':>8}   meaning")
found = {}
for entity in document.ents:
    found.setdefault(entity.label_, []).append(entity.text)
    print(f"{entity.text:<14}{entity.label_:>8}   {spacy.explain(entity.label_)}")
assert "Acme Corp" in found.get("ORG", []) and "1,250" in " ".join(found.get("MONEY", []))

text             label   meaning
Acme Corp          ORG   Companies, agencies, institutions, etc.
1,250            MONEY   Monetary values, including unit
3 March           DATE   Absolute or relative dates or periods
London             GPE   Countries, cities, states
Priya Shah      PERSON   People, including fictional
14 days           DATE   Absolute or relative dates or periods


**Reading the output.** The company, the amount (the model marks the number; the `$` is a separate token), the two dates, the place and the person — each labelled, each with its span. No rules were written; the model learned them from labelled news text.

**Into the CRM record.** Take the first of each type you need; flag the ticket if any is missing.

In [3]:
def crm_record(text: str) -> dict:
    record = {"company": None, "amount": None, "date": None}
    for entity in nlp(text).ents:
        if entity.label_ == "ORG" and record["company"] is None:
            record["company"] = entity.text
        elif entity.label_ == "MONEY" and record["amount"] is None:
            record["amount"] = entity.text
        elif entity.label_ == "DATE" and record["date"] is None:
            record["date"] = entity.text
    record["needs_review"] = None in record.values()
    return record


print(crm_record(ticket))
print(crm_record("Please refund me, this is unfair."))
assert (
    crm_record(ticket)["company"] == "Acme Corp" and crm_record("Please refund me.")["needs_review"]
)

{'company': 'Acme Corp', 'amount': '1,250', 'date': '3 March', 'needs_review': False}
{'company': None, 'amount': None, 'date': None, 'needs_review': True}


**The rule to remember.** NER is fast, cheap structured extraction for the standard entity types. For those types, it beats both regex and an LLM on cost; for custom types, an LLM with a schema (0.1 item 6) is easier than training.

| Use it when | Don't when | Instead use |
|---|---|---|
| people, organisations, money, dates, places at high volume | your entities are domain-specific (SKUs, drug names, error codes) | an LLM with a JSON schema; or a fine-tuned spaCy model |

**Watch out**
- Trained on news; on chat text with lowercase names and typos accuracy drops. Measure on your data.
- "Apple" the company vs the fruit is decided by context; short texts give little context.
- Never use NER alone to redact PII for compliance; it misses things. Combine with patterns and review.